# CDK2/CDK9/CDK1/CDK6/CDK5/CDK7の高活性化合物

## Why

`cdk20_similar_targets.ipynb` ranked CDK20's human paralogs by actual data
availability (not just sequence identity) and found six that are, unlike
CDK20 itself (zero PDB structures, ~1 usable ChEMBL activity), rich in both
solved structures and bioactivity data: `CDK2_HUMAN` (522 structures / 3454
activities), `CDK9_HUMAN` (28/2247), `CDK1_HUMAN` (14/1790), `CDK6_HUMAN`
(22/1053), `CDK5_HUMAN` (10/848), `CDK7_HUMAN` (56/824).

This notebook goes one level deeper on those six: pull each target's most
potent ChEMBL compounds. (A 3D-structure follow-up is deferred -- picking a
PDB structure by resolution alone isn't the right criterion, and there's no
clear approach yet for which structure(s) would actually be useful here.)

## Most potent compounds from ChEMBL

`chem.chembl.download_activities` with `normalize_smiles=True` (same
pattern as `example_chembl.ipynb`'s BRAF example) desalts/standardizes each
compound via the ChEMBL Structure Pipeline and aggregates duplicate
assay records into one row per unique compound
(`n`/`pchembl_mean`/`pchembl_median`/`pchembl_std`) -- exactly what's needed
to rank compounds by potency rather than by raw assay count. `mw=[250, 650]`
matches the same roughly drug-like range as that example. One tsv per
target, skipped on re-run if already downloaded.

In [ ]:
import os

from chem import chembl

TARGETS = ["CDK2_HUMAN", "CDK9_HUMAN", "CDK1_HUMAN", "CDK6_HUMAN", "CDK5_HUMAN", "CDK7_HUMAN"]
CHEMBL_OUTDIR = "cdk_paralogs_chembl_data"
os.makedirs(CHEMBL_OUTDIR, exist_ok=True)

for target in TARGETS:
    chembl.download_activities(
        target,
        mw=[250, 650],
        normalize_smiles=True,
        output=os.path.join(CHEMBL_OUTDIR, f"{target}.tsv"),
    )

### Compounds with pchembl_mean >= 8.8, per target

Each target's tsv filtered to `pchembl_mean >= 8.8` (roughly sub-2nM
potency) instead of a fixed top-N, sorted descending, with a `target`
column added so all six can be shown as one table -- some targets
contribute many rows above this bar, others just a few.

In [ ]:
import pandas as pd

PCHEMBL_THRESHOLD = 8.8

top_compounds = []
for target in TARGETS:
    df = pd.read_csv(os.path.join(CHEMBL_OUTDIR, f"{target}.tsv"), sep="\t")
    top = df[df["pchembl_mean"] >= PCHEMBL_THRESHOLD].sort_values("pchembl_mean", ascending=False).copy()
    top.insert(0, "target", target)
    top_compounds.append(top)

top_compounds_df = pd.concat(top_compounds, ignore_index=True)
print(f"{len(top_compounds_df)} compounds with pchembl_mean >= {PCHEMBL_THRESHOLD} across {len(TARGETS)} targets")
display(
    top_compounds_df[["target", "parent_chembl_id", "smiles", "n", "pchembl_mean", "mw"]]
    .style.hide(axis="index")
    .format({"pchembl_mean": "{:.2f}", "mw": "{:.2f}"})
)

### Top 5 structures per target, drawn

Same `MolsToGridImage` pattern as `example_chembl.ipynb`, one grid per
target.

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import MolsToGridImage

TOP_DRAW_N = 5

for target in TARGETS:
    top = top_compounds_df[top_compounds_df["target"] == target].head(TOP_DRAW_N)
    print(target)
    if top.empty:
        print(f"  no compounds with pchembl_mean >= {PCHEMBL_THRESHOLD}")
        continue
    mols = [Chem.MolFromSmiles(smi) for smi in top["smiles"]]
    legends = [f"{cid} pchembl_mean={v:.2f}" for cid, v in zip(top["parent_chembl_id"], top["pchembl_mean"])]
    display(MolsToGridImage(mols, molsPerRow=5, subImgSize=(220, 180), legends=legends))

## Summary

For all six data-rich CDK20 paralogs, this notebook now has, on disk and
ready for downstream SAR work: every ChEMBL compound with
`pchembl_mean >= 8.8` per target (`cdk_paralogs_chembl_data/`). A 3D-structure
follow-up (which PDB entries are actually useful, not just which have the
best resolution) is left for later, once there's a clearer idea of what to
pick them for.